# World Cup Transit Service near Levi's Stadium

In [ ]:
import warnings
warnings.filterwarnings("ignore")
                        
import altair as alt
import branca.colormap as cm
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils

credentials, _ = google.auth.default()

## Regional Trips

In [ ]:
levi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.bay_area_names,
    route_name_dict = wc_vars.special_bayarea_routes_dict,
    event_time_of_day_dict = wc_vars.levi_match_times  
)

In [ ]:
daily_trips_by_operator = D1.aggregate_daily_trips(
    levi_trips, ["service_date", "schedule_name"]
)

In [ ]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_operator, wc_vars.levi_dates, color_col="schedule_name"
).properties(
    title= "Daily Trips by Operator during World Cup",
    width=500, height=300
)

## Trips by Route

In [ ]:
daily_trips_by_route = D1.aggregate_daily_trips(
    levi_trips, ["service_date", "schedule_name", "route_name"]
)

In [ ]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_route, wc_vars.levi_dates, color_col="route_name"
).properties(
    title= "Daily Trips by Route during World Cup",
    width=500, height=300
)

In [ ]:
# Get each route's daily average number of trips on event days vs non-event days - plot on map
trips_by_event = D2.aggregate_by_event_type(
    levi_trips, 
    group_cols = ["schedule_name", "route_name", "direction_id", "event_day", "day_type"], 
    metric_cols = ["n_trips"]
).rename(columns = {
    "n_trips": "daily_trips", 
})

trips_wide = D2.make_wide(
    trips_by_event,
    index_cols=["schedule_name", "route_name", "direction_id"],
    pivot_cols=["day_type", "event_day"],
    value_cols=["daily_trips"],
)

trips_wide = trips_wide.assign(
    change_daily_trips = trips_wide[["change_daily_trips_weekday", "change_daily_trips_weekend"]].sum(axis=1),          
)

# Add route's line geometry
route_change_gdf = pd.merge(
    levi_trips[["schedule_name", "route_name", "direction_id", "geometry"]].drop_duplicates(),
    trips_wide,
    on = ["schedule_name", "route_name", "direction_id"],
    how = "inner"
)

In [ ]:
poi = gpd.read_parquet(
    f"{wc_vars.GCS_FILE_PATH}points_of_interest_{wc_vars.event_name}.parquet",
    storage_options = {"token": credentials.token},
    filters = [[("point_of_interest", "==", "Levi's Stadium")]]
)

In [ ]:
#D2.filter_fct_daily_scheduled_stops_to_special_routes
levi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes_keep_far_stops(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.bay_area_names,
    route_name_dict = wc_vars.special_bayarea_routes_dict,
    event_time_of_day_dict = wc_vars.levi_match_times
)

arrivals_wide = D2.stop_arrival_change_from_baseline_wide(levi_stop_arrivals)

In [ ]:
# check that our change column works with the shared legend cutoff across the maps in this notebook
#route_change_gdf.change_daily_trips.min(), route_change_gdf.change_daily_trips.max()

In [ ]:
# check that our change column works with the shared legend cutoff across the maps in this notebook
#arrivals_wide.combined_change_daily_arrivals.min(), arrivals_wide.combined_change_daily_arrivals.max()

In [ ]:
from _color_palette import SERVICE_CHANGE_COLORS, SERVICE_CHANGE_CAPTION
SERVICE_CHANGE_INDEX = [-200, -100, -1, 1, 50, 100, 200, 300, 800]

In [ ]:
route_map = route_change_gdf.explore(
    "change_daily_trips",
    tiles = "CartoDB Positron",
    cmap = cm.StepColormap(
        colors=SERVICE_CHANGE_COLORS, 
        index=SERVICE_CHANGE_INDEX, 
        vmin=route_change_gdf.change_daily_trips.min(), 
        vmax=route_change_gdf.change_daily_trips.max(), 
        tick_labels=SERVICE_CHANGE_INDEX,
        caption=SERVICE_CHANGE_CAPTION
    ),
    style_kwds = {
        "weight": 3
    },
    name = "Additional Trips (compared to baseline) on Special Service Routes" 
)

route_map = arrivals_wide.explore(
    "combined_change_daily_arrivals",
    m = route_map,
    cmap = cm.StepColormap(
        colors=SERVICE_CHANGE_COLORS, 
        index=SERVICE_CHANGE_INDEX, 
        vmin=arrivals_wide.combined_change_daily_arrivals.min(), 
        vmax=arrivals_wide.combined_change_daily_arrivals.max(), 
        tick_labels=SERVICE_CHANGE_INDEX,
        caption=SERVICE_CHANGE_CAPTION
    ),
    style_kwds={
        "style_function": lambda x: {
            "radius": x["properties"]["combined_change_daily_arrivals"]*0.05,
            "fillOpacity": 0.5,
            #"color": "#808080", # stroke color, if defined, stroke color will not be the color of the marker
            "strokeOpacity": 1,
            "weight": 2,
            #"weight":x["properties"]["combined_change_daily_arrivals"]*0.08, # weight seems to be stroke weight
        }
    },
    name = "Additional Arrivals (compared to baseline)" 
)

# https://python-visualization.github.io/folium/latest/user_guide/geojson/geojson_marker.html
# available colors, https://python-visualization.github.io/folium/latest/reference.html#folium.map.Marker
route_map = poi.explore(
    "point_of_interest",
    m=route_map,
    marker_type = "marker",
    name="Levi's Stadium",
    legend = False, # legend color is blue no matter what color is set for icon, confusing
    marker_kwds=dict(icon=folium.Icon(icon="star", color="darkpurple")),
)

folium.LayerControl().add_to(route_map)
route_map

## Stop Arrivals

In [ ]:
# filter to stops near stadium
levi_stop_arrivals_near = D2.filter_fct_daily_scheduled_stops_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.bay_area_names,
    route_name_dict = wc_vars.special_bayarea_routes_dict,
    event_time_of_day_dict = wc_vars.levi_match_times
)

arrivals_near_wide = D2.stop_arrival_change_from_baseline_wide(levi_stop_arrivals_near)

In [ ]:
operator_df = (
    arrivals_near_wide
    .groupby(["schedule_name", "route_id_array", "stop_name"])
    .agg({
        "change_daily_arrivals_weekday": "sum",
        "change_daily_arrivals_weekend": "sum",
        "stop_id": "nunique"
    })
    .reset_index()
    .rename(columns = {"stop_id": "n_stop_ids"})
)

In [ ]:
for i in sorted(operator_df.schedule_name.unique()):
    chart = chart_utils.weekday_weekend_chart_by_operator(operator_df, i)
    display(chart)